# CONEXIÓN CON EL STORAGE ACCOUNT

In [ ]:
storage_account = "name"
access_key = "key"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)

# LECTURA DE CSV "ENVIOS"

In [ ]:
ruta_csv = f"abfss://datalake@{storage_account}.dfs.core.windows.net/Bronze/envios/envios_logistica.csv"

df_envios = (
    spark.read
        .option("header", True)
        .option("delimiter", ";")
        .option("inferSchema", True)
        .csv(ruta_csv)
)
display(df_envios)

# LECTURA DE JSON "DISTANCIAS"


In [ ]:
# LECTURA DE JSON "DISTANCIAS"

ruta_json = f"abfss://datalake@{storage_account}.dfs.core.windows.net/Bronze/distancias/distancias.json"

df_distancias = (
    spark.read
        .option("multiLine", True)
        .json(ruta_json)
)

display(df_distancias)

# PERFILADO DE DATOS DEL CSV

In [ ]:
#Cantidad de registros
df_envios.count()

#ID repetido
df_envios.groupBy("id_envio").count().filter("count > 1").show()
from pyspark.sql.functions import col, count, when

# Valores nulos en cada columna
df_envios.select(]
    count(when(col(c).isNull(), c)).alias(c)
    for c in df envios.columns
]).show()

#Valores en columana 'estado'
df_envios.select("estado").distinct().show()


In [ ]:
#Creamos un DataFrame silver
df silver = df_envios

#Aca eliminamos registros que tengan valor nulo

df_silver = df_silver.dropna(
    subset=["fecha", "origen", "destino", "peso_kg"]
)

#Eliminamos duplicados
df_silver = df_silver.dropDuplicates(["id_envio"])



In [ ]:
#PRUEBA DE CONTROL
df_silver.count()

df_silver.groupBy("id_envio").count().filter("count > 1").show()

from pyspark.sql.functions import col, count, when

df_silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).show()

# JOIN DE LAS 2 FUENTES

In [ ]:
df_silver_join = df_silver.join(
    df_distancias,
    on=["origen", "destino"],
    how="left"
)

display(df_silver_join)
df_silver_join.printSchema()

GUARDADO EN LA CARPETA SILVER

In [ ]:
ruta_silver = f"abfss://datalake@{storage_account}.dfs.core.windows.net/Silver/joined"

df_silver_join.write
    .format("delta")
    .mode("overwrite")
    .save(ruta_silver)

In [ ]:
#CONTROL DE PRUEBAS
dbutils.fs.ls(ruta_silver)

df_silver_check = spark.read.format("delta").load(ruta_silver)

display(df_silver_check)